# NB08 – Remote Sensing Feature Extraction (Sentinel-2 via AGRS)

**Purpose**: Enrich every USABLE soil station with Sentinel-2 vegetation, water, and moisture
indices extracted from the Microsoft Planetary Computer STAC API using the AGRS library.

**Inputs**
- `data/intermediate/unified_surface_all.parquet` — 48 435 stations with coordinates

**Outputs**
- `data/intermediate/rs_cache/{iso3}.parquet` — per-country checkpoints (restart-safe)
- `data/intermediate/rs_features_all.parquet` — 48 435 stations × RS index columns

**Strategy**
- Process country by country, tiled on a 1° × 1° spatial grid (≈ Sentinel-2 scene footprint)
- 6 soil-relevant indices: NDVI, EVI, NDWI, NDMI, NDRE, GNDVI
- 3 seasonal snapshots per tile: `top_n_cloudfree` (n=3) within the country growing season
- 100 m buffer around each station point
- Per-country parquet checkpoints for restart safety

**Indices rationale**
| Index | Bands | Soil relevance |
|-------|-------|----------------|
| NDVI | B08/B04 | Vegetation density → organic matter, N |
| EVI | B08/B04/B02 | Biomass (less saturated than NDVI) |
| NDWI | B03/B08 | Surface water → soil moisture indicator |
| NDMI | B08/B11 | Canopy moisture → topsoil water content |
| NDRE | B08/B05 | Red-edge chlorophyll → N status |
| GNDVI | B08/B03 | Green NDVI → chlorophyll, CEC proxy |

In [ ]:
# ── Cell 1: Imports and configuration ──────────────────────────────────────────
import os
import logging
import warnings
from pathlib import Path

# Fix PROJ database conflict BEFORE any rasterio / pyproj imports.
# The base conda proj.db (version minor 2) is incompatible with pyproj ≥ 6.
# Point PROJ_DATA to pyproj's bundled proj.db (version minor 4).
_PROJ_DATA = Path('/home/agbelgaid/anaconda3/envs/agent/lib/python3.11'
                  '/site-packages/pyproj/proj_dir/share/proj')
os.environ['PROJ_DATA']   = str(_PROJ_DATA)
os.environ['PROJ_LIB']    = str(_PROJ_DATA)
os.environ['GDAL_DATA']   = str(Path('/home/agbelgaid/anaconda3/envs/agent'
                                     '/lib/python3.11/site-packages/rasterio/gdal_data'))

import numpy as np
import pandas as pd
import geopandas as gpd
from agrs.client import s2agc

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# ── Paths ──
BASE      = Path('/home/agbelgaid/Documents/WORKSPACE/DataCollection/SoilHive')
INTER     = BASE / 'data' / 'intermediate'
CACHE_DIR = INTER / 'rs_cache'
LOG_DIR   = BASE / 'logs'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ── Logger ──
LOG_FILE = LOG_DIR / 'nb08_remote_sensing.log'
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE, mode='w'),
        logging.StreamHandler(),
    ]
)
log = logging.getLogger(__name__)

# ── AGRS client ──
CLIENT = s2agc(source='planetary_computer', max_cloud=0.30)

# ── Indices to compute (soil-relevant subset) ──
RS_INDICES = ['NDVI', 'EVI', 'NDWI', 'NDMI', 'NDRE', 'GNDVI']

# ── Extraction parameters ──
POINT_BUFFER_M = 100     # metres — converted to degrees in make_geodataframe
TILE_DEG       = 1.0     # spatial tile size in degrees
N_SNAPSHOTS    = 3       # top N cloud-free scenes per tile
MAX_BATCH      = 150     # max stations per AGRS call

log.info('NB08 initialised — %d indices, %.0f m buffer, %.1f° tiles, PROJ_DATA=%s',
         len(RS_INDICES), POINT_BUFFER_M, TILE_DEG, os.environ['PROJ_DATA'])


In [2]:
# ── Cell 2: Load stations and define growing seasons ───────────────────────────

# Load unified dataset (all USABLE stations)
unified = pd.read_parquet(INTER / 'unified_surface_all.parquet')
stations = unified[['station_id', 'lat', 'lon', 'iso3', 'country']].copy()
log.info('Loaded %d stations across %d countries', len(stations), stations['iso3'].nunique())

# ── Growing seasons per country ──
# Format: (start_date, end_date) — 6-month window that captures peak vegetation
GROWING_SEASON = {
    # Southern hemisphere — austral summer (Oct–Apr)
    'AUS': ('2021-10-01', '2022-04-30'),
    'BWA': ('2021-10-01', '2022-04-30'),
    'AGO': ('2021-10-01', '2022-04-30'),
    'BDI': ('2021-10-01', '2022-04-30'),
    # West Africa / Sahel — boreal summer rainy season
    'BEN': ('2021-04-01', '2021-10-31'),
    'BFA': ('2021-05-01', '2021-10-31'),
    'CMR': ('2021-03-01', '2021-10-31'),
    'CAF': ('2021-04-01', '2021-10-31'),
    'TCD': ('2021-06-01', '2021-10-31'),
    # Mediterranean / North Africa — winter season
    'DZA': ('2021-01-01', '2021-06-30'),
    'MAR': ('2021-01-01', '2021-06-30'),
    'TUN': ('2021-01-01', '2021-06-30'),
    # Europe — spring/summer
    'ALB': ('2021-04-01', '2021-09-30'),
    'DEU': ('2021-04-01', '2021-09-30'),
    'EST': ('2021-05-01', '2021-09-30'),
    'FRA': ('2021-04-01', '2021-09-30'),
    'GEO': ('2021-04-01', '2021-09-30'),
    'GRC': ('2021-03-01', '2021-07-31'),
    'HRV': ('2021-04-01', '2021-09-30'),
    'MDA': ('2021-04-01', '2021-09-30'),
    'MNE': ('2021-04-01', '2021-09-30'),
    'NLD': ('2021-04-01', '2021-09-30'),
    # Tropical Pacific
    'PNG': ('2021-01-01', '2021-12-31'),
}

# Verify all countries are covered
missing_season = set(stations['iso3'].unique()) - set(GROWING_SEASON)
if missing_season:
    raise RuntimeError(f'Missing growing season definition for: {missing_season}')

log.info('Growing seasons defined for %d countries', len(GROWING_SEASON))

# Show season overview
season_df = pd.DataFrame(
    [(iso3, s, e) for iso3, (s, e) in GROWING_SEASON.items()],
    columns=['iso3', 'start', 'end']
).set_index('iso3')
print(season_df.to_string())

2026-03-20 01:25:43,056 [INFO] Loaded 48435 stations across 23 countries


2026-03-20 01:25:43,059 [INFO] Growing seasons defined for 23 countries


           start         end
iso3                        
AUS   2021-10-01  2022-04-30
BWA   2021-10-01  2022-04-30
AGO   2021-10-01  2022-04-30
BDI   2021-10-01  2022-04-30
BEN   2021-04-01  2021-10-31
BFA   2021-05-01  2021-10-31
CMR   2021-03-01  2021-10-31
CAF   2021-04-01  2021-10-31
TCD   2021-06-01  2021-10-31
DZA   2021-01-01  2021-06-30
MAR   2021-01-01  2021-06-30
TUN   2021-01-01  2021-06-30
ALB   2021-04-01  2021-09-30
DEU   2021-04-01  2021-09-30
EST   2021-05-01  2021-09-30
FRA   2021-04-01  2021-09-30
GEO   2021-04-01  2021-09-30
GRC   2021-03-01  2021-07-31
HRV   2021-04-01  2021-09-30
MDA   2021-04-01  2021-09-30
MNE   2021-04-01  2021-09-30
NLD   2021-04-01  2021-09-30
PNG   2021-01-01  2021-12-31


In [3]:
# ── Cell 3: Helper functions ───────────────────────────────────────────────────

def make_geodataframe(sub: pd.DataFrame, buffer_m: float) -> gpd.GeoDataFrame:
    """
    Convert a station DataFrame to a GeoDataFrame with a geographic degree buffer.
    Uses a fixed degree approximation — avoids UTM reprojection and PROJ database calls.
    At the equator: 1° ≈ 111 320 m → buffer_m / 111 320 gives buffer in degrees.
    Acceptable precision for Sentinel-2 10 m pixel sampling.
    """
    buffer_deg = buffer_m / 111_320.0
    gdf = gpd.GeoDataFrame(
        sub[['station_id', 'lat', 'lon']].copy(),
        geometry=gpd.points_from_xy(sub['lon'], sub['lat']),
        crs='EPSG:4326'
    )
    gdf['geometry'] = gdf.geometry.buffer(buffer_deg)
    return gdf


def tile_stations(sub: pd.DataFrame, tile_deg: float) -> list:
    """
    Split a station DataFrame into tile_deg × tile_deg spatial tiles.
    Returns a list of DataFrames, one per tile.
    """
    sub = sub.copy()
    sub['_tile_lat'] = np.floor(sub['lat'] / tile_deg).astype(int)
    sub['_tile_lon'] = np.floor(sub['lon'] / tile_deg).astype(int)
    tiles = []
    for _, tile_df in sub.groupby(['_tile_lat', '_tile_lon']):
        tiles.append(tile_df.drop(columns=['_tile_lat', '_tile_lon']).reset_index(drop=True))
    return tiles


def pivot_features(features_df: pd.DataFrame, field_id_col: str) -> pd.DataFrame:
    """
    Pivot AGRS features_df (long: one row per field × fraction) to wide format.
    Output columns:
      rs_{INDEX}_season_mean / _std / _median  — aggregate across all snapshots
      rs_{INDEX}_f{fraction_pct}               — value at each temporal fraction
    """
    if features_df.empty:
        return pd.DataFrame(columns=[field_id_col])

    index_cols = [c for c in features_df.columns
                  if c not in (field_id_col, 'fraction', 'snapshot_date', 'eo:cloud_cover')]

    records = []
    for fid, fdf in features_df.groupby(field_id_col):
        row = {field_id_col: fid}
        for idx_col in index_cols:
            vals = fdf[idx_col].dropna()
            if vals.empty:
                row[f'rs_{idx_col}_season_mean']   = np.nan
                row[f'rs_{idx_col}_season_std']    = np.nan
                row[f'rs_{idx_col}_season_median'] = np.nan
            else:
                row[f'rs_{idx_col}_season_mean']   = float(vals.mean())
                row[f'rs_{idx_col}_season_std']    = float(vals.std())
                row[f'rs_{idx_col}_season_median'] = float(vals.median())
            # Per-fraction values
            for _, frow in fdf.iterrows():
                frac = frow.get('fraction', None)
                if frac is None:
                    continue
                frac_pct = int(round(float(frac) * 100))
                row[f'rs_{idx_col}_f{frac_pct:02d}'] = frow[idx_col]
        records.append(row)

    return pd.DataFrame(records)


log.info('Helper functions defined — degree buffer: %.6f° (≈ %.0f m)',
         100 / 111_320.0, 100.0)


2026-03-20 01:25:43,091 [INFO] Helper functions defined — degree buffer: 0.000898° (≈ 100 m)


In [4]:
# ── Cell 4: Per-country extraction loop ────────────────────────────────────────

COUNTRIES = sorted(stations['iso3'].unique())
errors = []
skipped_tiles = []

for iso3 in COUNTRIES:
    checkpoint = CACHE_DIR / f'{iso3}.parquet'
    if checkpoint.exists():
        log.info('  %s: checkpoint found, skipping', iso3)
        continue

    country_df   = stations[stations['iso3'] == iso3].copy()
    start_date, end_date = GROWING_SEASON[iso3]
    n_stations   = len(country_df)
    log.info('  %s: %d stations | season %s → %s', iso3, n_stations, start_date, end_date)

    # Spatial tiling
    tiles = tile_stations(country_df, TILE_DEG)
    log.info('  %s: %d tiles (%.1f° grid)', iso3, len(tiles), TILE_DEG)

    country_results = []

    for t_idx, tile_df in enumerate(tiles):
        # Further split oversized tiles
        if len(tile_df) > MAX_BATCH:
            batches = [tile_df.iloc[i:i + MAX_BATCH]
                       for i in range(0, len(tile_df), MAX_BATCH)]
        else:
            batches = [tile_df]

        for b_idx, batch_df in enumerate(batches):
            try:
                gdf = make_geodataframe(batch_df, POINT_BUFFER_M)

                features_df = CLIENT.get_features(
                    fields            = gdf,
                    field_id_col      = 'station_id',
                    start_date        = start_date,
                    end_date          = end_date,
                    snapshot_strategy = 'top_n_cloudfree',
                    n_snapshots       = N_SNAPSHOTS,
                    indices           = RS_INDICES,
                    return_mode       = 'features',
                )

                if not features_df.empty:
                    wide = pivot_features(features_df, 'station_id')
                    country_results.append(wide)
                    log.info(
                        '    tile %d/%d batch %d/%d: %d stations → %d features rows',
                        t_idx + 1, len(tiles), b_idx + 1, len(batches),
                        len(batch_df), len(wide)
                    )
                else:
                    log.warning(
                        '    tile %d/%d batch %d/%d: no features returned (no scenes?)',
                        t_idx + 1, len(tiles), b_idx + 1, len(batches)
                    )
                    skipped_tiles.append({'iso3': iso3, 'tile': t_idx, 'batch': b_idx,
                                          'reason': 'empty_features'})

            except Exception as exc:
                log.warning(
                    '    tile %d/%d batch %d/%d: ERROR — %s',
                    t_idx + 1, len(tiles), b_idx + 1, len(batches), exc
                )
                errors.append({'iso3': iso3, 'tile': t_idx, 'batch': b_idx, 'error': str(exc)})

    # Save country checkpoint
    if country_results:
        country_wide = pd.concat(country_results, ignore_index=True)
        # Fill any station not returned by AGRS with NaN row
        country_wide = country_df[['station_id']].merge(country_wide, on='station_id', how='left')
        country_wide.to_parquet(checkpoint, index=False)
        rs_cols = [c for c in country_wide.columns if c.startswith('rs_')]
        coverage = country_wide[rs_cols[0]].notna().mean() if rs_cols else 0.0
        log.info('  %s: saved checkpoint — %d stations, %.1f%% RS coverage',
                 iso3, len(country_wide), coverage * 100)
    else:
        # All tiles failed — save empty checkpoint to allow restart
        empty = country_df[['station_id']].copy()
        empty.to_parquet(checkpoint, index=False)
        log.warning('  %s: all tiles failed — saved empty checkpoint', iso3)

log.info('Extraction loop complete | %d tile errors | %d skipped tiles',
         len(errors), len(skipped_tiles))

2026-03-20 01:25:43,287 [INFO]   AGO: 1571 stations | season 2021-10-01 → 2022-04-30


2026-03-20 01:25:43,313 [INFO]   AGO: 85 tiles (1.0° grid)


Fields:   0%|          | 0/6 [00:00<?, ?it/s]

2026-03-20 01:25:50,356 [WARNING] CPLE_AppDefined in PROJ: internal_proj_create_from_database: /home/agbelgaid/anaconda3/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 3 is expected. It comes from another PROJ installation.


2026-03-20 01:25:50,357 [WARNING] CPLE_AppDefined in The definition of projected CRS EPSG:32733 got from GeoTIFF keys is not the same as the one from the EPSG registry, which may cause issues during reprojection operations. Set GTIFF_SRS_SOURCE configuration option to EPSG to use official parameters (overriding the ones from GeoTIFF keys), or to GEOKEYS to use custom values from GeoTIFF keys and drop the EPSG code.


...

2026-03-20 01:40:19,117 [INFO] GDAL signalled an error: err_no=1, msg='PROJ: internal_proj_create_from_database: /home/agbelgaid/anaconda3/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 3 is expected. It comes from another PROJ installation.'


Fields:   0%|          | 0/1 [00:00<?, ?it/s]


2026-03-20 01:40:19,121 [WARNING]     tile 12/12 batch 1/1: ERROR — The EPSG code is unknown. PROJ: internal_proj_create_from_database: /home/agbelgaid/anaconda3/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 3 is expected. It comes from another PROJ installation.


2026-03-20 01:40:19,124 [WARNING]   TUN: all tiles failed — saved empty checkpoint


2026-03-20 01:40:19,124 [INFO] Extraction loop complete | 1179 tile errors | 0 skipped tiles


In [5]:
# ── Cell 5: Merge checkpoints → single parquet ─────────────────────────────────

checkpoints = sorted(CACHE_DIR.glob('*.parquet'))
log.info('Merging %d country checkpoints...', len(checkpoints))

frames = []
for cp in checkpoints:
    iso3 = cp.stem
    df = pd.read_parquet(cp)
    df['iso3'] = iso3
    frames.append(df)

rs_all = pd.concat(frames, ignore_index=True)

# Ensure all original stations are present (left-join on unified)
rs_all = stations[['station_id', 'iso3']].merge(
    rs_all.drop(columns=['iso3'], errors='ignore'),
    on='station_id', how='left'
)

rs_cols  = [c for c in rs_all.columns if c.startswith('rs_')]
n_filled = rs_all[rs_cols[0]].notna().sum() if rs_cols else 0
log.info('Merged: %d stations | %d RS feature columns | %.1f%% coverage',
         len(rs_all), len(rs_cols), 100 * n_filled / len(rs_all))

out_path = INTER / 'rs_features_all.parquet'
rs_all.to_parquet(out_path, index=False)
log.info('Saved rs_features_all.parquet — %.1f MB',
         out_path.stat().st_size / 1e6)

2026-03-20 01:40:19,131 [INFO] Merging 23 country checkpoints...


2026-03-20 01:40:19,498 [INFO] Merged: 48435 stations | 0 RS feature columns | 0.0% coverage


2026-03-20 01:40:19,522 [INFO] Saved rs_features_all.parquet — 0.3 MB


In [6]:
# ── Cell 6: Coverage report ─────────────────────────────────────────────────────

import io, sys

rs_cols = [c for c in rs_all.columns if c.startswith('rs_')]

# Per-country RS coverage
coverage_rows = []
for iso3, grp in rs_all.groupby('iso3'):
    n_total = len(grp)
    if rs_cols:
        n_with_rs = grp[rs_cols[0]].notna().sum()
    else:
        n_with_rs = 0
    coverage_rows.append({
        'iso3':      iso3,
        'n_stations': n_total,
        'n_with_rs':  n_with_rs,
        'coverage_%': round(100 * n_with_rs / n_total, 1),
    })

coverage_df = pd.DataFrame(coverage_rows).sort_values('n_stations', ascending=False)
print('=== RS Coverage by Country ===')
print(coverage_df.to_string(index=False))
print()

# Global stats
total = len(rs_all)
filled = rs_all[rs_cols[0]].notna().sum() if rs_cols else 0
print(f'TOTAL stations : {total:,}')
print(f'With RS data   : {filled:,} ({100*filled/total:.1f}%)')
print(f'RS feature cols: {len(rs_cols)}')
if rs_cols:
    print(f'Feature columns (sample): {rs_cols[:8]}')

# Error summary
if errors:
    print(f'\n⚠  Tile errors: {len(errors)}')
    for e in errors[:10]:
        print(f'  {e["iso3"]} tile={e["tile"]} batch={e["batch"]}: {e["error"][:80]}')

log.info('NB08 complete — %d stations, %d RS feature columns, %.1f%% coverage',
         total, len(rs_cols), 100 * filled / total)
log.info('NB08 completed successfully — proceed to NB09 (append RS to country CSVs)')

2026-03-20 01:40:19,616 [INFO] NB08 complete — 48435 stations, 0 RS feature columns, 0.0% coverage


2026-03-20 01:40:19,616 [INFO] NB08 completed successfully — proceed to NB09 (append RS to country CSVs)


=== RS Coverage by Country ===
iso3  n_stations  n_with_rs  coverage_%
 AUS       33748          0         0.0
 FRA        2962          0         0.0
 DEU        2873          0         0.0
 BFA        1671          0         0.0
 AGO        1571          0         0.0
 CMR        1323          0         0.0
 NLD        1287          0         0.0
 BWA        1253          0         0.0
 BEN         716          0         0.0
 GRC         305          0         0.0
 EST         202          0         0.0
 MAR         114          0         0.0
 CAF          80          0         0.0
 HRV          79          0         0.0
 ALB          67          0         0.0
 TUN          40          0         0.0
 MDA          35          0         0.0
 BDI          28          0         0.0
 MNE          24          0         0.0
 GEO          18          0         0.0
 PNG          16          0         0.0
 TCD          13          0         0.0
 DZA          10          0         0.0

TOTAL st